# Flujos de trabajo: cuando fijar el camino sale mejor

El [cuaderno de orquestación](orquestacion.ipynb) montó un sistema enrutado y descubrió que se quedaba clavado en la fiabilidad de su enrutador. Aquel sistema era, mirado de cerca, **un flujo de trabajo**: un clasificador y un especialista, en ese orden, decidido por nosotros.

Este cuaderno coge esa idea y la pone frente a su alternativa. El mismo triaje de la secretaría, montado dos veces:

* Como **agente**: tres herramientas, y el modelo decide cuál llama y cuándo para.
* Como **flujo**: clasificar, consultar y redactar, en ese orden, siempre.

El [capítulo](https://iraitzm.github.io/manual-ia-generativa/parts/agentes/flujos.html) afirma cuatro cosas sobre esa comparación. Aquí se comprueban las cuatro con números, y **dos salen distintas de lo que promete el titular**:

1. El flujo cuesta menos y es más predecible.
2. Un flujo fija **el camino**, no la salida.
3. Un flujo no degrada bien: falla con la misma seguridad con la que acierta.
4. El híbrido, flujo por defecto y agente cuando hay duda, reparte el gasto.

Merece la pena ejecutarlo antes de leer las conclusiones de cada sección.

## Preparación

In [ ]:
!pip install -q duckdb "transformers>=4.51" torch

In [ ]:
import pathlib
import subprocess
import sys

LOCAL = pathlib.Path("../../data/secretaria")
COLAB = pathlib.Path("manual-ia-generativa/data/secretaria")

if LOCAL.exists():
    base = LOCAL
else:
    if not COLAB.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--quiet",
             "https://github.com/IraitzM/manual-ia-generativa.git"],
            check=True,
        )
    base = COLAB

sys.path.insert(0, str(base.resolve()))

from secretaria import preparar

ctx = preparar()
con = ctx.conectar()
ALUMNO = "A2023000"

In [ ]:
import json
import re

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

tok = AutoTokenizer.from_pretrained(ctx.modelo)
modelo = AutoModelForCausalLM.from_pretrained(ctx.modelo, dtype=torch.float32)
modelo.eval()

PATRON = re.compile(r"<tool_call>\s*(\{.*?\})\s*</tool_call>", re.S)

## Las tres herramientas

Son código y consultan de verdad: dos van a DuckDB y la tercera busca en el corpus. Ninguna de las tres sabe nada de arquitecturas, y esa es la gracia: **las mismas herramientas van a servir al agente y al flujo**. Lo único que cambia entre las dos arquitecturas es quién decide cuál se llama.

In [ ]:
TRAMITES = [r[0] for r in con.execute("select distinct tramite from dim_plazo").fetchall()]


def consultar_plazo(tramite: str) -> str:
    fila = con.execute(
        "select tramite, fecha_inicio, fecha_fin from dim_plazo "
        "where tramite ilike ? limit 1",
        [f"%{tramite}%"],
    ).fetchone()
    if not fila:
        return f"No hay ningún plazo llamado {tramite!r}."
    return f"Plazo {fila[0]}: del {fila[1]} al {fila[2]}."


def consultar_expediente(asignatura: str = "") -> str:
    filas = con.execute(
        """
        select a.asignatura, c.convocatoria, c.nota
        from fct_matriculas m
        join dim_asignatura a on a.asignatura_id = m.asignatura_id
        left join fct_calificaciones c on c.matricula_id = m.matricula_id
        where m.alumno_id = ? and a.asignatura ilike ?
        order by a.asignatura limit 8
        """,
        [ALUMNO, f"%{asignatura}%"],
    ).fetchall()
    if not filas:
        return "Sin matrículas que encajen."
    return "; ".join(f"{f[0]}: {f[2] if f[2] is not None else 'sin nota'}" for f in filas)


DOCS = {d["id"]: d["texto"] for d in ctx.documentos()}


def buscar_normativa(consulta: str) -> str:
    palabras = re.findall(r"\w{5,}", consulta.lower())
    mejor, puntos = "", 0
    for texto in DOCS.values():
        for parrafo in texto.split("\n\n"):
            p = sum(parrafo.lower().count(w) for w in palabras)
            if p > puntos:
                mejor, puntos = parrafo, p
    return mejor[:300] if mejor else "Nada en la normativa."


HERRAMIENTAS = {
    "consultar_plazo": consultar_plazo,
    "consultar_expediente": consultar_expediente,
    "buscar_normativa": buscar_normativa,
}

print(consultar_plazo("defensa_tfg"))
print(consultar_expediente("Cálculo"))

In [ ]:
def esquema(nombre, descripcion, propiedades, obligatorios):
    return {"type": "function", "function": {
        "name": nombre, "description": descripcion,
        "parameters": {"type": "object", "properties": propiedades,
                       "required": obligatorios}}}


ESQUEMAS = [
    esquema("consultar_plazo", "Fechas de inicio y fin de un trámite administrativo.",
            {"tramite": {"type": "string", "description": ", ".join(TRAMITES)}}, ["tramite"]),
    esquema("consultar_expediente", "Asignaturas, notas y convocatorias del alumno que pregunta.",
            {"asignatura": {"type": "string"}}, []),
    esquema("buscar_normativa", "Busca una regla o un requisito en la normativa académica.",
            {"consulta": {"type": "string"}}, ["consulta"]),
]

SISTEMA = ("Eres el asistente de la secretaría académica. Para responder debes llamar "
           "a una de las herramientas disponibles. No respondas de memoria.")

## Los dieciocho casos

Etiquetados por la fuente que hay que consultar, que es lo que se puede comprobar de forma objetiva. Siete de plazos, seis de expediente y cinco de normativa.

Los de plazo llevan además el **trámite exacto** que debería consultarse, porque acertar la fuente y equivocarse de trámite produce una respuesta con fecha, muy convincente y falsa.

In [ ]:
CASOS = [
    ("¿cuándo empieza el plazo de matrícula ordinaria?", "plazo", "matricula_ordinaria"),
    ("¿hasta qué día puedo pedir la anulación de convocatoria?", "plazo", "anulacion_convocatoria"),
    ("¿qué fechas tiene la defensa del trabajo de fin de grado?", "plazo", "defensa_tfg"),
    ("¿cuándo se puede solicitar el cambio de grupo?", "plazo", "cambio_grupo"),
    ("¿hasta cuándo está abierta la beca general?", "plazo", "solicitud_beca_general"),
    ("¿qué día acaba el plazo de reconocimiento de créditos?", "plazo", "reconocimiento_creditos"),
    ("¿cuándo son los exámenes de la convocatoria ordinaria del primer semestre?",
     "plazo", "convocatoria_ordinaria_s1"),
    ("¿qué nota saqué en cálculo?", "expediente", "Cálculo"),
    ("¿tengo aprobada estadística?", "expediente", "Estadística"),
    ("¿en qué asignaturas estoy matriculado?", "expediente", ""),
    ("¿cuál es mi nota de bases de datos?", "expediente", "Bases de datos"),
    ("¿he aprobado álgebra lineal?", "expediente", "Álgebra"),
    ("¿qué asignaturas llevo suspensas?", "expediente", ""),
    ("¿qué requisitos piden para la beca general?", "normativa", ""),
    ("¿cuántas convocatorias tengo por asignatura?", "normativa", ""),
    ("¿se puede repetir un examen ya aprobado?", "normativa", ""),
    ("¿qué nota mínima hace falta para matrícula de honor?", "normativa", ""),
    ("¿cómo se calcula el recargo de la matrícula extraordinaria?", "normativa", ""),
]

CATEGORIAS = ["plazo", "expediente", "normativa"]
CAT_A_TOOL = {"plazo": "consultar_plazo",
              "expediente": "consultar_expediente",
              "normativa": "buscar_normativa"}

print(f"{len(CASOS)} casos etiquetados")

## El flujo: tres pasos, siempre los mismos

**Paso 1, clasificar.** Una sola pasada del modelo con la decodificación restringida a las tres categorías, la técnica del [cuaderno de prompting](../contexto/prompting.ipynb). Devuelve también su confianza, que hará falta al final.

**Paso 2, consultar.** Sin modelo. El argumento sale por solapamiento de palabras y la consulta la hace el código.

**Paso 3, redactar.** Una generación con los datos ya delante.

Dos llamadas al modelo por consulta. Ni una más, pase lo que pase.

In [ ]:
PRIMEROS = {c: tok(c, add_special_tokens=False).input_ids[0] for c in CATEGORIAS}
IDS = torch.tensor([PRIMEROS[c] for c in CATEGORIAS])

SISTEMA_CLASIFICADOR = """Clasifica la consulta del alumno según qué fuente hay que consultar:
- plazo: pregunta por una fecha o un plazo administrativo.
- expediente: pregunta por sus propias notas o matrículas.
- normativa: pregunta por una regla o un requisito general.
Devuelve únicamente el identificador.

Ejemplos:
¿cuándo empieza el plazo de reconocimiento? -> plazo
¿tengo aprobada física? -> expediente
¿qué nota mínima piden para matrícula de honor? -> normativa
¿qué día es la defensa del trabajo? -> plazo
¿cuántos créditos llevo superados? -> expediente
¿se puede repetir un examen aprobado? -> normativa"""


def clasificar(consulta):
    """Paso 1. Una pasada, sin generar texto. Devuelve categoría y confianza."""
    texto = tok.apply_chat_template(
        [{"role": "system", "content": SISTEMA_CLASIFICADOR},
         {"role": "user", "content": consulta}],
        tokenize=False, add_generation_prompt=True, enable_thinking=False)
    entrada = tok(texto, return_tensors="pt")
    with torch.no_grad():
        logits = modelo(**entrada).logits[0, -1]
    prob = torch.softmax(logits[IDS], dim=-1)
    i = int(prob.argmax())
    return CATEGORIAS[i], float(prob[i]), int(entrada.input_ids.shape[1])


def extraer_argumento(categoria, consulta):
    """Paso 2a. Sin modelo: solapamiento de palabras."""
    if categoria == "plazo":
        palabras = set(re.findall(r"\w{4,}", consulta.lower()))
        mejor, puntos = TRAMITES[0], -1
        for t in TRAMITES:
            p = len(palabras & set(t.lower().split("_")))
            if p > puntos:
                mejor, puntos = t, p
        return {"tramite": mejor}
    if categoria == "expediente":
        for (asignatura,) in con.execute("select asignatura from dim_asignatura").fetchall():
            if asignatura.lower()[:6] in consulta.lower():
                return {"asignatura": asignatura}
        return {"asignatura": ""}
    return {"consulta": consulta}


def redactar(consulta, datos, muestrear=False):
    """Paso 3. Una generación con los datos delante."""
    texto = tok.apply_chat_template(
        [{"role": "system", "content": "Responde en una frase usando solo los datos aportados."},
         {"role": "user", "content": f"Consulta: {consulta}\nDatos: {datos}"}],
        tokenize=False, add_generation_prompt=True, enable_thinking=False)
    entrada = tok(texto, return_tensors="pt")
    with torch.no_grad():
        salida = modelo.generate(
            **entrada, max_new_tokens=64, do_sample=muestrear,
            temperature=0.7 if muestrear else None,
            top_p=0.9 if muestrear else None,
            pad_token_id=tok.eos_token_id)
    respuesta = tok.decode(salida[0][entrada.input_ids.shape[1]:], skip_special_tokens=True)
    return respuesta.strip(), int(entrada.input_ids.shape[1])


def flujo(consulta, muestrear=False):
    """El camino lo decidimos nosotros. Siempre estos tres pasos."""
    categoria, confianza, t1 = clasificar(consulta)
    args = extraer_argumento(categoria, consulta)
    datos = HERRAMIENTAS[CAT_A_TOOL[categoria]](**args)
    respuesta, t2 = redactar(consulta, datos, muestrear)
    return {"traza": [CAT_A_TOOL[categoria]], "categoria": categoria,
            "confianza": confianza, "args": args, "respuesta": respuesta,
            "tokens": t1 + t2, "llamadas": 2}

## El agente: el bucle del primer cuaderno

Las mismas tres herramientas, pero declaradas al modelo. El número de llamadas ya no lo sabemos de antemano: depende de lo que decida.

In [ ]:
def agente(consulta, max_vueltas=3, muestrear=False):
    """El camino lo decide el modelo. Es el bucle de bucle-a-mano.ipynb."""
    mensajes = [{"role": "system", "content": SISTEMA},
                {"role": "user", "content": consulta}]
    traza, tokens, llamadas, args = [], 0, 0, {}
    respuesta = ""
    for _ in range(max_vueltas):
        texto = tok.apply_chat_template(
            mensajes, tools=ESQUEMAS, tokenize=False,
            add_generation_prompt=True, enable_thinking=False)
        entrada = tok(texto, return_tensors="pt")
        tokens += int(entrada.input_ids.shape[1])
        llamadas += 1
        with torch.no_grad():
            salida = modelo.generate(
                **entrada, max_new_tokens=96, do_sample=muestrear,
                temperature=0.7 if muestrear else None,
                top_p=0.9 if muestrear else None,
                pad_token_id=tok.eos_token_id)
        texto_salida = tok.decode(salida[0][entrada.input_ids.shape[1]:],
                                  skip_special_tokens=True)
        m = PATRON.search(texto_salida)
        if not m:
            respuesta = texto_salida.strip()
            break
        try:
            peticion = json.loads(m.group(1))
        except json.JSONDecodeError:
            respuesta = texto_salida.strip()
            break
        nombre = peticion.get("name", "")
        args = peticion.get("arguments", {}) or {}
        traza.append(nombre)
        if nombre not in HERRAMIENTAS:
            mensajes.append({"role": "assistant", "content": texto_salida})
            mensajes.append({"role": "user", "content": f"No existe {nombre}."})
            continue
        try:
            resultado = HERRAMIENTAS[nombre](**args)
        except TypeError as exc:
            resultado = f"Error de argumentos: {exc}"
        mensajes.append({"role": "assistant", "content": texto_salida})
        mensajes.append({"role": "user", "content": f"Resultado: {resultado}"})
    return {"traza": traza, "args": args, "respuesta": respuesta,
            "tokens": tokens, "llamadas": llamadas}

## Las dos arquitecturas, medidas

Tarda unos minutos. El modelo es diminuto y va en CPU.

In [ ]:
def medir(fn, casos):
    ruta_ok = tokens = llamadas = arg_ok = arg_total = 0
    fallos = []
    for consulta, categoria, arg_esperado in casos:
        r = fn(consulta)
        esperada = CAT_A_TOOL[categoria]
        acierta = bool(r["traza"]) and r["traza"][0] == esperada
        ruta_ok += acierta
        tokens += r["tokens"]
        llamadas += r["llamadas"]
        if categoria == "plazo" and arg_esperado:
            arg_total += 1
            arg_ok += acierta and r["args"].get("tramite", "") == arg_esperado
        if not acierta:
            fallos.append((consulta, r["traza"], esperada))
    return {"ruta_ok": ruta_ok, "n": len(casos), "tokens": tokens,
            "llamadas": llamadas, "arg_ok": arg_ok, "arg_total": arg_total,
            "fallos": fallos}


m_flujo = medir(flujo, CASOS)
m_agente = medir(agente, CASOS)

print(f"{'arquitectura':14s} {'ruta':>10s} {'tokens':>9s} {'llamadas':>9s} {'trámite':>10s}")
print("-" * 56)
for nombre, m in [("flujo", m_flujo), ("agente", m_agente)]:
    print(f"{nombre:14s} {m['ruta_ok']:>4d}/{m['n']:<5d} {m['tokens']:>9d} "
          f"{m['llamadas']:>9d} {m['arg_ok']:>4d}/{m['arg_total']:<5d}")

El flujo gana en las dos columnas que importan, y no por poco: acierta la fuente en 17 de los 18 casos frente a 11 del agente, gastando alrededor de un tercio de los tokens.

Conviene entender **por qué**, porque no es que fijar el camino tenga magia. Es que las dos arquitecturas no están resolviendo el mismo problema:

* El flujo elige **entre tres etiquetas**, con la decodificación restringida a tres tokens y seis ejemplos delante. Es casi un clasificador.
* El agente tiene que **emitir un JSON bien formado** eligiendo entre tres esquemas, acertando el nombre de la función y el de sus argumentos, todo en la misma generación.

Es la aritmética del cuaderno anterior vista desde el otro lado: repartir compensa cuando cada paso es más fácil que el problema entero, y aquí separar "qué fuente" de "cómo se consulta" hace el primer paso mucho más fácil.

**El coste, además, se sabe de antemano.** Dos llamadas por consulta, siempre, y una de ellas sin generar texto. Que el agente haya acabado en el mismo número de llamadas es una casualidad de estos casos, no una propiedad suya: nadie podía escribirlo en el presupuesto antes de ejecutarlo.

**Y ahora la columna que baja los humos.** En acertar el *trámite* concreto, las dos empatan. Fijar el camino ha arreglado el paso que decide la fuente y no ha tocado el que extrae el argumento, que sigue siendo un solapamiento de palabras bastante tosco. Acertar la fuente y fallar el trámite produce una respuesta con una fecha real dentro, sacada de la tabla, y completamente equivocada. Es la peor clase de error que existe: parece verificada.

In [ ]:
print("Fallos del flujo:")
for consulta, traza, esperada in m_flujo["fallos"]:
    print(f"  {consulta[:50]:52s} {str(traza):32s} esperada={esperada}")

print("\nFallos del agente:")
for consulta, traza, esperada in m_agente["fallos"]:
    print(f"  {consulta[:50]:52s} {str(traza):32s} esperada={esperada}")

## Lo que un flujo fija de verdad

Aquí se comprueba la afirmación del capítulo que más se malinterpreta: **un flujo no hace determinista la salida, hace determinista el camino**.

La prueba es directa. El mismo caso cinco veces, con muestreo activado en las dos arquitecturas, contando dos cosas por separado: cuántos **caminos** distintos salen y cuántos **textos** distintos salen.

In [ ]:
REPETIR = 5
muestra = [CASOS[0][0], CASOS[7][0], CASOS[13][0]]

print(f"{'consulta':44s} {'arq':7s} {'caminos':>8s} {'textos':>7s}")
print("-" * 70)
for consulta in muestra:
    for nombre, fn in [("flujo", flujo), ("agente", agente)]:
        caminos, textos = set(), set()
        for _ in range(REPETIR):
            r = fn(consulta, muestrear=True)
            caminos.add(tuple(r["traza"]))
            textos.add(r["respuesta"])
        print(f"{consulta[:42]:44s} {nombre:7s} {len(caminos):>8d} {len(textos):>7d}")

La mitad del resultado es la esperada y la otra mitad no, así que conviene mirarlas por separado.

**Lo esperado:** la columna de caminos del flujo es **1** en las tres consultas y la de textos no lo es. Eso es exactamente lo que se compra. La frase cambia entre ejecuciones porque al final sigue habiendo un modelo generativo redactando, y eso no se arregla fijando pasos. Lo que no cambia nunca es **qué fuente se consultó**, que es lo que se audita, lo que se factura y lo que hay que poder afirmar delante de quien pregunte de dónde salió un dato.

**Lo que no se esperaba:** el camino del agente también ha salido único en las tres. Con temperatura 0,7 y tres herramientas, la elección resultó ser robusta al muestreo. Conviene decirlo tal cual en lugar de esconderlo, porque el resultado cómodo habría sido el contrario.

Y sin embargo la conclusión del capítulo no se cae, se afila. La diferencia entre las dos arquitecturas no es que una varíe y la otra no. Es **cómo se sabe**:

* Que el flujo tiene un solo camino se demuestra **leyendo el código**. Vale para todas las consultas, incluidas las que nunca habéis probado.
* Que el agente tenga un solo camino solo se sabe **midiéndolo**, con estas tres consultas, con este modelo, con esta temperatura y con estas tres herramientas. Cambiad cualquiera de las cuatro cosas y hay que volver a medir.

Una es una garantía y la otra es una observación. Cuando alguien de cumplimiento pregunte si el sistema siempre consulta la fuente oficial, esa distinción es la respuesta entera.

## Los casos que no caben en el dibujo

Hasta aquí el flujo va ganando. Toca la parte incómoda.

Estas cuatro consultas necesitan **dos** fuentes: un plazo y el expediente. El flujo, por construcción, consulta una.

In [ ]:
COMPUESTOS = [
    ("¿me da tiempo a pedir la anulación de convocatoria de cálculo?", ["plazo", "expediente"]),
    ("¿puedo presentarme a la extraordinaria de estadística y estoy a tiempo?",
     ["plazo", "expediente"]),
    ("con mi nota de bases de datos, ¿me interesa pedir revisión y cuándo acaba el plazo?",
     ["plazo", "expediente"]),
    ("¿estoy matriculado en el TFG y cuándo tengo que defenderlo?", ["plazo", "expediente"]),
]

for consulta, fuentes in COMPUESTOS:
    rf = flujo(consulta)
    ra = agente(consulta)
    print(f"\n{consulta}")
    print(f"  necesita  : {fuentes}")
    print(f"  flujo     : {rf['traza']}")
    print(f"    -> {rf['respuesta'][:130]}")
    print(f"  agente    : {ra['traza']}")
    print(f"    -> {ra['respuesta'][:130]}")

Mirad las respuestas, no las trazas. Y no solo las del flujo.

**Las dos arquitecturas fallan las cuatro consultas, y ninguna avisa.** No salta una excepción, no aparece un campo vacío, no hay una frase del tipo "me falta consultar tu expediente". Las dos obtienen media respuesta y redactan una frase completa con el tono de siempre.

Merece la pena leer despacio lo que produce el flujo, porque es peor que fallar:

* *"La nota de bases de datos indica que el plazo finaliza en 6.0."* Una frase sin sentido construida con un dato real.
* *"Estoy matriculado en el TFG y tengo que defenderlo el 25 de octubre."* Esa fecha **no existe en ninguna tabla**. El flujo consultó el expediente, no tenía ningún plazo delante, y el modelo rellenó el hueco.

El agente no lo hace mejor. Se conforma con una fuente igual que el flujo, y encima acierta menos la que elige: en uno de los casos consulta el plazo de `matricula_extraordinaria` para responder cuándo se defiende un TFG, y entrega esas fechas tan tranquilo.

Así que el resultado honesto es que **el bucle no salva estos casos**. Lo que separa a las dos arquitecturas aquí no es el resultado, que es igual de malo, sino la naturaleza del fallo:

* En el agente es una **decisión**: tenía vueltas disponibles y herramientas declaradas, y eligió parar. Eso se ataca con el prompt, con más ejemplos o con un modelo mejor.
* En el flujo es la **forma del dibujo**: no había ningún camino que consultara dos fuentes. Ningún prompt lo arregla; hay que dibujar otro flujo.

Esto es lo que el capítulo llama **no degradar bien**, y aplica a los dos. El precio de fijar el camino no es la rigidez, que se ve venir y se acepta. Es que el sistema no sabe que se ha quedado corto, así que tampoco lo dice. El ejercicio 2 va sobre eso.

## El híbrido

La salida del capítulo: el flujo por defecto y el agente para lo que el clasificador no ve claro. La pieza que lo hace posible es que `clasificar` ya devolvía su confianza.

Primero conviene mirar cómo se reparte esa confianza, porque de eso depende que el umbral sirva de algo.

In [ ]:
confianzas = sorted(clasificar(c)[1] for c, _, _ in CASOS)
print(f"confianza del clasificador: mín {confianzas[0]:.2f}  "
      f"mediana {confianzas[len(confianzas) // 2]:.2f}  máx {confianzas[-1]:.2f}")
print("distribución:", " ".join(f"{c:.2f}" for c in confianzas))

In [ ]:
print(f"{'umbral':>7s} {'ruta':>10s} {'tokens':>9s} {'al agente':>10s}")
print("-" * 40)
for umbral in [0.0, 0.5, 0.7, 0.9, 1.0]:
    ruta_ok = tokens = escapes = 0
    for consulta, categoria, _ in CASOS:
        _, conf, t1 = clasificar(consulta)
        if conf < umbral:
            escapes += 1
            r = agente(consulta)
            tokens += t1 + r["tokens"]
        else:
            r = flujo(consulta)
            tokens += r["tokens"]
        ruta_ok += bool(r["traza"]) and r["traza"][0] == CAT_A_TOOL[categoria]
    print(f"{umbral:>7.1f} {ruta_ok:>4d}/{len(CASOS):<5d} {tokens:>9d} {escapes:>10d}")

Esta tabla es la que hay que mirar con más calma, porque va en contra de lo que promete el patrón.

Cuanto más se sube el umbral, más consultas se derivan al agente, **más se gasta y menos se acierta**. La versión híbrida no mejora al flujo en ningún punto: lo empeora de forma monótona hasta convertirse en el agente, con el clasificador pagado encima.

No es que el híbrido sea mala idea. Es que **una salida de emergencia solo sirve si da a un sitio mejor**, y en este montaje no lo da: la primera tabla ya nos dijo que el agente acierta 11 de 18 y el flujo 17. Derivar hacia el agente solo podía estropear las cosas, y el umbral lo único que decide es cuánto.

Es la misma lección del enrutador del cuaderno de orquestación, vista desde el otro lado. Allí el sistema no podía superar a su peor eslabón; aquí, conectar una pieza peor a una mejor arrastra al conjunto hacia la peor. **Antes de componer dos piezas hay que medir las dos por separado**, y esa medida es justo la que casi nunca se hace porque el patrón parece razonable sobre el papel.

Hay además un motivo de fondo por el que el umbral no podía salvarlo: la confianza mide **seguridad, no acierto**. Un clasificador puede estar al 0,98 y equivocarse, y entonces el escape ni siquiera se activa. Es la misma trampa del juez del [cuaderno de evaluación](../produccion/evaluacion.ipynb), donde la señal útil tampoco estaba donde parecía.

¿Cuándo sí compensa, entonces? Cuando lo que hay al otro lado de la puerta resuelve casos que el camino barato no puede resolver, como los `COMPUESTOS` de la sección anterior. Ese es el ejercicio 3.

## Lo mismo, declarado con Agno

Todo lo de arriba son funciones de Python llamándose una detrás de otra. El capítulo sostiene que un framework de flujos no viene a sustituir ese `if`, sino a poder meterse entre los pasos.

Vale la pena declarar el mismo flujo con Agno para ver qué cambia y qué no. Los ejecutores son **las mismas funciones**: aquí no hay ningún agente de por medio, y eso evita de paso el fallo silencioso que encontró el [cuaderno de frameworks](frameworks.ipynb) al enchufar un modelo local al atajo del framework.

In [ ]:
!pip install -q agno

In [ ]:
from agno.workflow.router import Router
from agno.workflow.step import Step, StepInput, StepOutput
from agno.workflow.workflow import Workflow


def paso_clasificar(paso: StepInput) -> StepOutput:
    categoria, _, _ = clasificar(paso.input)
    return StepOutput(content=categoria)


def _responder(paso: StepInput, categoria: str) -> StepOutput:
    args = extraer_argumento(categoria, paso.input)
    datos = HERRAMIENTAS[CAT_A_TOOL[categoria]](**args)
    respuesta, _ = redactar(paso.input, datos)
    return StepOutput(content=respuesta)


paso_plazo = Step(name="plazo", executor=lambda p: _responder(p, "plazo"))
paso_expediente = Step(name="expediente", executor=lambda p: _responder(p, "expediente"))
paso_normativa = Step(name="normativa", executor=lambda p: _responder(p, "normativa"))

DESTINOS = {"plazo": paso_plazo, "expediente": paso_expediente, "normativa": paso_normativa}


def derivar(paso: StepInput):
    return [DESTINOS[paso.previous_step_content]]


flujo_agno = Workflow(
    name="triaje de secretaría",
    steps=[
        Step(name="clasificar", executor=paso_clasificar),
        Router(name="derivar", selector=derivar, choices=list(DESTINOS.values())),
    ],
)

r = flujo_agno.run(input="¿cuándo empieza el plazo de matrícula ordinaria?")
print("respuesta:", r.content[:200])


def mostrar(salidas, sangria=2):
    """Los pasos ejecutados. El Router anida dentro la rama que eligió."""
    for s in salidas or []:
        print(f"{' ' * sangria}{s.step_name:12s} -> {str(s.content)[:60]}")
        mostrar(getattr(s, "steps", None), sangria + 2)


print("\npasos ejecutados:")
mostrar(r.step_results)

Comparad ese bloque con la función `flujo` de más arriba. Hace lo mismo y ocupa el triple.

Lo que ha aparecido a cambio es la última celda: **la lista de pasos ejecutados, con su nombre y su salida, y la rama que el `Router` eligió anidada dentro**, sin haber escrito una sola línea para registrarla. Ese es el trato. Declarar los pasos en lugar de llamarlos es lo que permite que algo se meta entre ellos, y una vez que ese algo existe puede ser una traza, un punto de guardado, una reanudación tras un fallo o una parada para que lo apruebe una persona.

Con `db=` configurada, `Workflow` persiste la sesión y el flujo se puede reanudar donde se quedó. Eso, y no el `Router`, es lo que justifica la dependencia. Para tres funciones que terminan en el mismo proceso donde empezaron, el `if` seguía siendo mejor ingeniería.

## Ejercicios

**1. Arreglad el paso que decide.** Es el mismo consejo que dejó el cuaderno de orquestación y aquí es más barato de seguir, porque en el flujo la decisión está en una sola función con su prompt. Mejorad `SISTEMA_CLASIFICADOR` y volved a medir las dos arquitecturas. ¿Cuál de las dos mejora más?

**2. Que el flujo sepa que se ha quedado corto.** Añadid un cuarto paso que compruebe si la respuesta contiene un dato de cada fuente que la consulta pedía, y que devuelva "necesito consultar también el expediente" en lugar de una frase segura. Medidlo sobre `COMPUESTOS`. Convertir un fallo silencioso en uno visible vale más que casi cualquier otra mejora.

**3. Una salida de emergencia que valga la pena.** El híbrido salió mal porque daba a un sitio peor. Arregladlo: construid el flujo de dos consultas (clasificar, consultar plazo, consultar expediente, redactar) y derivad hacia **él**, en lugar de hacia el agente, cuando el clasificador dude. Medid las dos cosas: si arregla los `COMPUESTOS` y **cuánto empeora los simples** al consultar siempre de más. Esa segunda medida es la que decide.

**4. El umbral que os conviene.** Con la tabla del híbrido delante, poned precio a las dos columnas: cuánto cuesta un token y cuánto cuesta una respuesta equivocada en vuestro caso. El umbral óptimo sale de esa cuenta, no de mirar la tabla.

**5. La reproducibilidad con temperatura 0.** Repetid la prueba de los caminos con `muestrear=False` en las dos arquitecturas. El agente pasa a ser reproducible también, y conviene entender por qué eso no arregla el problema: la traza sigue dependiendo del contenido que devuelvan las herramientas, que cambia cuando cambian los datos.

**6. Lo mismo en una plataforma.** Montad este flujo en n8n o en Sim y cronometrad las dos cosas: lo que tardáis en tenerlo funcionando, y lo que tardáis en tener el conjunto de dieciocho casos ejecutándose contra él de forma automática. La distancia entre esos dos números es el argumento entero del capítulo.

## Lo que os lleváis

* **El flujo ganó en acierto y en coste**, 17 de 18 contra 11 y un tercio de los tokens. El motivo no es que fijar el camino tenga magia: es que separar "qué fuente" de "cómo se consulta" convierte el primer paso en un problema mucho más fácil que elegir un esquema y rellenarlo bien.
* **El coste de un flujo se conoce de antemano y el de un agente no.** Que aquí coincidieran en número de llamadas fue una casualidad de estos casos, no una propiedad.
* **Fijar el camino arregla el paso que decide y no toca los demás.** El trámite concreto se seguía fallando igual en las dos arquitecturas, 4 de 7. Las respuestas con fecha real y equivocada salen de ahí.
* **Un flujo fija el camino, no la redacción.** Y la diferencia con el agente no es que uno varíe y el otro no, porque aquí ninguno varió: es que una propiedad se demuestra **leyendo el código** y la otra hay que **medirla** cada vez que algo cambia.
* **Ninguna de las dos degrada bien.** Ante un caso que no cabía, las dos contestaron a medias sin avisar, y el flujo llegó a inventarse una fecha de defensa que no está en ninguna tabla.
* **El híbrido salió peor y más caro.** Una salida de emergencia solo sirve si da a un sitio mejor, y eso se sabe midiendo las dos piezas por separado antes de conectarlas.
* **Un framework de flujos se paga por lo que se mete entre los pasos**, no por el `Router`. Si no necesitáis trazas, persistencia ni reanudación, el `if` gana.

Y el remate del capítulo, que estos números respaldan: antes de montar un agente, mirad qué fracción de vuestro tráfico real cabe en un dibujo. Suele ser mucha más de la que se supone.

Queda la frontera con la persona: [la interfaz del agente](interfaz.ipynb).